In [2]:
import pandas as pd

In [29]:
description = pd.read_excel("../data/Task_Data_Scientist_Dataset.xlsx", sheet_name="Description")
sd = pd.read_excel("../data/Task_Data_Scientist_Dataset.xlsx", sheet_name="Soc_Dem")
products = pd.read_excel("../data/Task_Data_Scientist_Dataset.xlsx", sheet_name="Products_ActBalance")
flows = pd.read_excel("../data/Task_Data_Scientist_Dataset.xlsx", sheet_name="Inflow_Outflow")
targets = pd.read_excel("../data/Task_Data_Scientist_Dataset.xlsx", sheet_name="Sales_Revenues")

### Social Demographic

In [5]:
sd.info()

<class 'pandas.DataFrame'>
RangeIndex: 1615 entries, 0 to 1614
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Client  1615 non-null   int64
 1   Sex     1612 non-null   str  
 2   Age     1615 non-null   int64
 3   Tenure  1615 non-null   int64
dtypes: int64(3), str(1)
memory usage: 52.4 KB


In [6]:
sd[sd["Sex"].isna()]

,Client,Sex,Age,Tenure
422,1363,NaN,38,8
1062,953,NaN,50,35
1494,843,NaN,36,151


In [46]:
sd["Sex"] = sd["Sex"].fillna("UNK")

In [16]:
sd["Client"].nunique()

1615

In [38]:
sd.describe()

,Client,Age,Tenure
count,1615.000000,1615.000000,1615.000000
mean,808.000000,42.848916,101.339938
std,466.354658,18.550529,64.917297
min,1.000000,0.000000,0.000000
25%,404.500000,29.000000,44.000000
50%,808.000000,41.000000,97.000000
75%,1211.500000,57.000000,151.000000
max,1615.000000,97.000000,273.000000


In [47]:
missing = sd[~sd.Client.isin(flows.Client)]
df = missing.merge(products, on="Client")
print(df[["Age", "Tenure", "Count_CA", "ActBal_CA"]].describe())
print(df.merge(targets, on="Client")[["Sale_MF", "Sale_CC", "Sale_CL"]].mean())  # vs overall rates

             Age      Tenure   Count_CA     ActBal_CA
count  28.000000   28.000000  28.000000     28.000000
mean   47.285714  106.857143   1.107143   1616.773763
std    15.402879   57.010535   0.416270   4426.306707
min    23.000000    0.000000   1.000000      0.000000
25%    36.750000   70.500000   1.000000     52.724286
50%    45.500000  107.500000   1.000000    406.657500
75%    56.750000  150.250000   1.000000   1151.913839
max    77.000000  235.000000   3.000000  23516.171429
Sale_MF    0.000000
Sale_CC    0.277778
Sale_CL    0.055556
dtype: float64


In [48]:
df = (sd.merge(products, on="Client")
         .merge(targets, on="Client", how="left"))
df["no_flow_data"] = ~df.Client.isin(flows.Client)

# 1) Profile comparison
cols = ["Age", "Tenure", "Count_CA", "ActBal_CA"]
print(df.groupby("no_flow_data")[cols].agg(["mean", "median"]).T)

# 2) Product ownership (NaN = not owned)
own = df[["Count_SA", "Count_MF", "Count_OVD", "Count_CC", "Count_CL"]].notna()
print(own.groupby(df.no_flow_data).mean())

# 3) Sale rates, with n of labeled clients
lab = df[df.Sale_CC.notna()]
print(lab.groupby("no_flow_data")[["Sale_MF", "Sale_CC", "Sale_CL"]].agg(["mean", "sum", "count"]))

no_flow_data            False        True 
Age       mean      42.770636    47.285714
          median    41.000000    45.500000
Tenure    mean     101.242596   106.857143
          median    96.000000   107.500000
Count_CA  mean       1.078135     1.107143
          median     1.000000     1.000000
ActBal_CA mean    2453.101745  1616.773763
          median   464.607143   406.657500
              Count_SA  Count_MF  Count_OVD  Count_CC  Count_CL
no_flow_data                                                   
False              1.0       1.0        1.0       1.0       1.0
True               1.0       1.0        1.0       1.0       1.0
               Sale_MF                Sale_CC                Sale_CL         \
                  mean    sum count      mean    sum count      mean    sum   
no_flow_data                                                                  
False         0.202944  193.0   951  0.249211  237.0   951  0.303891  289.0   
True          0.000000    0.0    18  0.27

In [49]:
prod_cols = ["Count_SA", "Count_MF", "Count_OVD", "Count_CC", "Count_CL"]
own = df[prod_cols].fillna(0) > 0
print(own.groupby(df.no_flow_data).mean())

              Count_SA  Count_MF  Count_OVD  Count_CC  Count_CL
no_flow_data                                                   
False         0.268431  0.192187    0.26402   0.10712  0.085066
True          0.000000  0.035714    0.00000   0.00000  0.000000


In [50]:
lab = df[df.Sale_CC.notna()].copy()
for p in ["MF", "CC", "CL"]:
    owns = lab[f"Count_{p}"].fillna(0) > 0
    print(p, pd.crosstab(owns, lab[f"Sale_{p}"], margins=True), "\n")

MF Sale_MF   0.0  1.0  All
Count_MF               
False     642  143  785
True      134   50  184
All       776  193  969 

CC Sale_CC   0.0  1.0  All
Count_CC               
False     643  216  859
True       84   26  110
All       727  242  969 

CL Sale_CL   0.0  1.0  All
Count_CL               
False     629  261  890
True       50   29   79
All       679  290  969 



In [51]:
for p in ["MF", "CC", "CL"]:
    r = targets.loc[targets[f"Sale_{p}"] == 1, f"Revenue_{p}"]
    nr = targets.loc[targets[f"Sale_{p}"] == 0, f"Revenue_{p}"]
    print(p, "buyers:", r.describe().round(1).to_dict(), "| non-buyer revenue != 0:", (nr != 0).sum())

MF buyers: {'count': 193.0, 'mean': 9.7, 'std': 20.6, 'min': 0.0, '25%': 1.2, '50%': 3.1, '75%': 9.1, 'max': 220.5} | non-buyer revenue != 0: 0
CC buyers: {'count': 242.0, 'mean': 10.9, 'std': 34.2, 'min': 0.1, '25%': 2.2, '50%': 4.4, '75%': 7.0, 'max': 408.0} | non-buyer revenue != 0: 0
CL buyers: {'count': 290.0, 'mean': 12.0, 'std': 9.9, 'min': 1.1, '25%': 6.2, '50%': 10.7, '75%': 15.6, 'max': 133.3} | non-buyer revenue != 0: 0


In [53]:
buy = targets.merge(products, on="Client").merge(sd, on="Client")
for p in ["MF", "CC", "CL"]:
    r = buy.loc[buy[f"Sale_{p}"] == 1, f"Revenue_{p}"]
    top5 = r.nlargest(int(len(r) * 0.05)).sum() / r.sum()
    print(p, "share of revenue from top 5% buyers:", round(top5, 2),
          "| zero-revenue buyers:", (r == 0).sum())

# is revenue related to anything obvious?
num = buy.select_dtypes("number").fillna(0)
for p in ["MF", "CC", "CL"]:
    b = num[buy[f"Sale_{p}"] == 1]
    print(p, b.corr(method="spearman")[f"Revenue_{p}"].drop(
        [c for c in b.columns if "Revenue" in c or "Sale" in c or c == "Client"]
    ).abs().nlargest(4).round(2).to_dict())

MF share of revenue from top 5% buyers: 0.38 | zero-revenue buyers: 0
CC share of revenue from top 5% buyers: 0.54 | zero-revenue buyers: 0
CL share of revenue from top 5% buyers: 0.15 | zero-revenue buyers: 0
MF {'ActBal_MF': 0.09, 'Count_MF': 0.07, 'Count_CA': 0.07, 'Count_CC': 0.04}
CC {'ActBal_CA': 0.1, 'Count_SA': 0.1, 'ActBal_MF': 0.09, 'ActBal_SA': 0.09}
CL {'ActBal_CA': 0.18, 'Count_CC': 0.12, 'Count_CL': 0.09, 'ActBal_CL': 0.08}


### Products

In [ ]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 1615 entries, 0 to 1614
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Client      1615 non-null   int64  
 1   Count_CA    1615 non-null   int64  
 2   Count_SA    426 non-null    float64
 3   Count_MF    306 non-null    float64
 4   Count_OVD   419 non-null    float64
 5   Count_CC    170 non-null    float64
 6   Count_CL    135 non-null    float64
 7   ActBal_CA   1615 non-null   float64
 8   ActBal_SA   426 non-null    float64
 9   ActBal_MF   306 non-null    float64
 10  ActBal_OVD  419 non-null    float64
 11  ActBal_CC   170 non-null    float64
 12  ActBal_CL   135 non-null    float64
dtypes: float64(11), int64(2)
memory usage: 164.2 KB


In [41]:
products = products.fillna(0)

In [43]:
products["has_SA"] = products["Count_SA"] > 0
products["has_MF"] = products["Count_MF"] > 0
products["has_OVD"] = products["Count_OVD"] > 0
products["has_CC"] = products["Count_CC"] > 0
products["has_CL"] = products["Count_CL"] > 0


In [44]:
products

,Client,Count_CA,Count_SA,Count_MF,Count_OVD,Count_CC,Count_CL,ActBal_CA,ActBal_SA,ActBal_MF,ActBal_OVD,ActBal_CC,ActBal_CL,has_SA,has_MF,has_OVD,has_CC,has_CL
0,1,1,0.0,0.0,1.0,0.0,0.0,1333.802857,0.000000,0.000000,0.0,0.000000,0.0,False,False,True,False,False
1,2,1,1.0,0.0,0.0,0.0,0.0,110.768571,13326.190357,0.000000,0.0,0.000000,0.0,True,False,False,False,False
2,3,1,0.0,1.0,0.0,0.0,0.0,482.654643,0.000000,107.251786,0.0,0.000000,0.0,False,True,False,False,False
3,4,1,1.0,0.0,0.0,1.0,0.0,1599.840714,76.437143,0.000000,0.0,1110.381786,0.0,True,False,False,True,False
4,5,1,1.0,0.0,0.0,0.0,0.0,5353.483929,8079.715714,0.000000,0.0,0.000000,0.0,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1610,1611,1,0.0,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,False,False,True,False,False
1611,1612,1,0.0,0.0,1.0,0.0,1.0,170.593214,0.000000,0.000000,0.0,0.000000,6250.0,False,False,True,False,True
1612,1613,1,0.0,0.0,0.0,0.0,0.0,676.008571,0.000000,0.000000,0.0,0.000000,0.0,False,False,False,False,False
1613,1614,1,0.0,0.0,0.0,0.0,0.0,0.010357,0.000000,0.000000,0.0,0.000000,0.0,False,False,False,False,False


In [39]:
products.describe()

,Client,Count_CA,Count_SA,Count_MF,Count_OVD,Count_CC,Count_CL,ActBal_CA,ActBal_SA,ActBal_MF,ActBal_OVD,ActBal_CC,ActBal_CL
count,1615.000000,1615.000000,426.000000,306.000000,419.0,170.0,135.000000,1615.000000,426.000000,306.000000,419.000000,170.000000,135.000000
mean,808.000000,1.078638,1.166667,4.676471,1.0,1.0,1.185185,2438.601941,15201.520284,20517.533479,126.449897,351.295420,4237.641981
std,466.354658,0.333035,0.468205,7.845009,0.0,0.0,0.520980,7072.777350,32365.183430,78046.994833,290.573266,486.679857,4167.134365
min,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.000000,0.000000,0.000000,0.000000,0.000000,-15.479286,0.000000
25%,404.500000,1.000000,1.000000,1.000000,1.0,1.0,1.000000,61.562143,728.145000,0.000000,0.000000,1.241071,1020.163393
50%,808.000000,1.000000,1.000000,2.000000,1.0,1.0,1.000000,462.221786,4413.503571,1700.943036,0.000000,153.006071,3141.900000
75%,1211.500000,1.000000,1.000000,5.000000,1.0,1.0,1.000000,2174.864286,14622.666071,11169.181339,93.114286,545.008839,5851.657679
max,1615.000000,4.000000,5.000000,79.000000,1.0,1.0,5.000000,171575.889643,389883.830714,761235.504286,2055.325357,3522.233571,20749.294643


In [17]:
products["Client"].nunique()

1615

### Flows

In [9]:
flows.info()

<class 'pandas.DataFrame'>
RangeIndex: 1587 entries, 0 to 1586
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Client                        1587 non-null   int64  
 1   VolumeCred                    1587 non-null   float64
 2   VolumeCred_CA                 1587 non-null   float64
 3   TransactionsCred              1587 non-null   int64  
 4   TransactionsCred_CA           1587 non-null   int64  
 5   VolumeDeb                     1587 non-null   float64
 6   VolumeDeb_CA                  1587 non-null   float64
 7   VolumeDebCash_Card            1587 non-null   float64
 8   VolumeDebCashless_Card        1587 non-null   float64
 9   VolumeDeb_PaymentOrder        1587 non-null   float64
 10  TransactionsDeb               1587 non-null   int64  
 11  TransactionsDeb_CA            1587 non-null   int64  
 12  TransactionsDebCash_Card      1587 non-null   int64  
 13  TransactionsDe

In [ ]:
flows['Client'].max()

np.int64(1615)

In [18]:
flows["Client"].nunique()

1587

In [ ]:
flows['Client'].max() - len(flows) # transactionless clients

np.int64(28)

In [11]:
flows

,Client,VolumeCred,VolumeCred_CA,TransactionsCred,TransactionsCred_CA,VolumeDeb,VolumeDeb_CA,VolumeDebCash_Card,VolumeDebCashless_Card,VolumeDeb_PaymentOrder,TransactionsDeb,TransactionsDeb_CA,TransactionsDebCash_Card,TransactionsDebCashless_Card,TransactionsDeb_PaymentOrder
0,1017,0.005000,0.005000,1,1,118.046429,118.046429,25.000000,91.796429,0.000000,12,12,2,9,0
1,1456,364.772143,364.752857,4,3,774.980000,774.980000,14.285714,497.444286,258.607143,28,28,1,20,5
2,545,2471.719286,2471.719286,14,14,2190.257143,2190.257143,303.571429,179.082143,1229.428571,23,23,3,5,12
3,667,1127.792857,1127.792857,4,4,1344.603571,1344.603571,642.857143,194.285714,249.321429,21,21,3,4,8
4,1381,571.618214,571.618214,3,3,57.071429,57.071429,10.714286,46.357143,0.000000,6,6,1,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1582,108,1175.480357,1171.326071,8,6,465.500000,465.500000,0.000000,31.607143,430.500000,10,10,0,2,7
1583,1079,281.812857,281.812857,2,2,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0
1584,1126,283.104643,283.104643,2,2,40.285714,40.285714,0.000000,0.000000,39.285714,2,2,0,0,1
1585,134,528.711071,433.429286,3,2,403.821429,401.000000,285.714286,0.000000,0.000000,6,5,2,0,0


In [ ]:
(flows['TransactionsDeb'] >= flows['TransactionsDeb_CA']).value_counts() # add automatic parent and child check

True    1587
Name: count, dtype: int64

In [35]:
PARENT_CHILD_MAP = {
    "VolumeCred": ["VolumeCred_CA"],
    "TransactionsCred": ["TransactionsCred_CA"],
    "VolumeDeb": ["VolumeDeb_CA", "VolumeDebCash_Card", "VolumeDebCashless_Card", "VolumeDeb_PaymentOrder"],
    "TransactionsDeb": ["TransactionsDeb_CA", "TransactionsDebCash_Card", "TransactionsDebCashless_Card", "TransactionsDeb_PaymentOrder"],
}

In [36]:
def validate_parent_dependencies(df: pd.DataFrame, mapping: dict[str, list[str]], raise_error: bool = False) -> dict:
    """Checks that all child columns are less than or equal to their parent column."""
    violations = {}

    for parent, children in mapping.items():
        if parent not in df.columns:
            continue
            
        for child in children:
            if child not in df.columns:
                continue
                
            # Find rows where child strictly exceeds parent
            failing_rows = df[df[child] > df[parent]]
            
            if not failing_rows.empty:
                violations[f"{child} > {parent}"] = failing_rows[[parent, child]]

    if violations and raise_error:
        raise ValueError(f"Found {len(violations)} column relationship violations.")
        
    return violations

In [37]:
validate_parent_dependencies(flows, PARENT_CHILD_MAP)

{}

### targets

In [30]:
targets.info()

<class 'pandas.DataFrame'>
RangeIndex: 969 entries, 0 to 968
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Client      969 non-null    int64  
 1   Sale_MF     969 non-null    int64  
 2   Sale_CC     969 non-null    int64  
 3   Sale_CL     969 non-null    int64  
 4   Revenue_MF  969 non-null    float64
 5   Revenue_CC  969 non-null    float64
 6   Revenue_CL  969 non-null    float64
dtypes: float64(3), int64(4)
memory usage: 53.1 KB


In [52]:
targets

,Client,Sale_MF,Sale_CC,Sale_CL,Revenue_MF,Revenue_CC,Revenue_CL
0,1094,0,1,0,0.000000,5.138571,0.000000
1,1484,1,0,1,25.609107,0.000000,8.722500
2,20,0,1,0,0.000000,3.822500,0.000000
3,1193,0,0,0,0.000000,0.000000,0.000000
4,142,0,0,0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
964,1316,0,0,1,0.000000,0.000000,3.317500
965,905,1,1,1,1.761071,0.892857,13.166786
966,938,0,1,1,0.000000,4.928571,7.812143
967,434,0,0,0,0.000000,0.000000,0.000000


In [31]:
len(sd) * .6

969.0

In [32]:
targets["Client"].nunique()

969

## Phase 0

In [57]:
prod_cols = [c for c in products.columns if c != "Client"]
flags = ["SA", "MF", "OVD", "CC", "CL"]

master = (sd.merge(products, on="Client", how="left")
             .merge(flows, on="Client", how="left")
             .merge(targets, on="Client", how="left"))

master["no_flow_data"] = ~master.Client.isin(flows.Client)
master["is_labeled"] = master.Client.isin(targets.Client)
master[prod_cols] = master[prod_cols].fillna(0)
for f in flags:
    master[f"has_{f}"] = (master[f"Count_{f}"] > 0).astype(int)
master["Sex"] = master["Sex"].fillna("Unknown")

assert master.Client.is_unique and len(master) == 1615
master.to_parquet("../data/processed/master.parquet", index=False)

# what to paste back
print(master.shape, master.is_labeled.sum(), master.no_flow_data.sum())
print(master.isna().sum()[lambda s: s > 0])
print(master.dtypes.value_counts())
print(master.Sex.value_counts())
print(master.head(3).T)

(1615, 43) 969 28
VolumeCred                       28
VolumeCred_CA                    28
TransactionsCred                 28
TransactionsCred_CA              28
VolumeDeb                        28
VolumeDeb_CA                     28
VolumeDebCash_Card               28
VolumeDebCashless_Card           28
VolumeDeb_PaymentOrder           28
TransactionsDeb                  28
TransactionsDeb_CA               28
TransactionsDebCash_Card         28
TransactionsDebCashless_Card     28
TransactionsDeb_PaymentOrder     28
Sale_MF                         646
Sale_CC                         646
Sale_CL                         646
Revenue_MF                      646
Revenue_CC                      646
Revenue_CL                      646
dtype: int64
float64    31
int64       9
bool        2
str         1
Name: count, dtype: int64
Sex
M      856
F      756
UNK      3
Name: count, dtype: int64
                                        0            1            2
Client                              